# GroupDNA: WhatsApp Chat Analyzer 🧬
**Built with Python Fundamentals & NumPy**

This notebook parses raw WhatsApp export files to generate a behavioural analytics report.
**Technical Constraints:**
* No `pandas` for data manipulation.
* No `matplotlib` for visualizations (rendered entirely in text).
* No `re` (regex) for parsing.
* No `collections.Counter` for frequency mapping.


In [16]:

# 1. SETUP & CONFIGURATION

import numpy as np
from datetime import datetime, timedelta
import string
# We define a custom set of stop words since we cannot use NLTK or external NLP libraries.
# Using a 'set' provides O(1) lookup time, which is much faster than a list.

STOP_WORDS={'i', 'is', 'the', 'a', 'and', 'or', 'to', 'of', 'in', 'on', 'for', 'it',
    'that', 'you', 'my', 'me', 'with', 'this', 'but', 'are', 'not', 'have',
    'at', 'be', 'we', 'so', 'was', 'if', 'do', 'just', 'like', 'can', 'what',
    'im', 'as', 'your', 'its', 'all', 'out', 'from', 'they', 'how', 'up'}

# Dictionaries for our personality archetype keywords
CARING_KEYWORDS = ['okay', 'safe', 'eat', 'sleep', 'take care', 'are you', 'please', 'reminder', 'drink water', 'forget']
LAUGH_KEYWORDS = ['lol', 'lmao', 'haha', 'rofl', 'hahaha', 'lmfao', 'hehe']

## Feature 1: The Chat Parser
Real-world data is messy. This parser reads the text file line-by-line and explicitly handles:
1. **System Messages:** (e.g., "User left")
2. **Media Omitted:** (Images/Videos replaced by text)
3. **Deleted Messages:** (Skipped entirely)
4. **Multi-line Messages:** (Reconstructed without timestamps)

*Note: Since regex is forbidden, we use string heuristics (checking specific index positions for slashes) to identify timestamp lines.*

In [17]:

# 2. THE DATA PARSER

def parse_chat(filepath):
    messages = []
    participants = []
    stats = {'system': 0, 'media': 0, 'deleted': 0}

    # Using 'with open' natively handles file closure and memory management
    with open(filepath, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    current_msg = None

    for line in lines:
        line = line.strip()
        if not line:
            continue # Skip empty lines silently

        # --- HEURISTIC DATE CHECK ---
        # Instead of regex, we check if the line is long enough and has slashes
        # in the expected date format positions (e.g., DD/MM/YY)
        is_new_message = False
        if len(line) >= 8 and (line[2] == '/' or line[1] == '/') and (line[5] == '/' or line[4] == '/'):
            is_new_message = True

        if is_new_message:
            if current_msg:
                messages.append(current_msg) # Save the previous built message

            # Split the line into timestamp and the rest of the message
            # We use try/except to gracefully catch unexpected format variations
            try:
                parts = line.split(' - ', 1)
                if len(parts) < 2: continue

                date_str = parts[0].replace(',', '').strip()
                rest = parts[1]

                # If there's no colon, it's a system message generated by WhatsApp
                if ':' not in rest:
                    stats['system'] += 1
                    current_msg = None
                    continue

                sender, text = rest.split(':', 1)
                sender = sender.strip()
                text = text.strip()

                is_media = "<Media omitted>" in text
                is_deleted = "This message was deleted" in text

                if is_media: stats['media'] += 1
                if is_deleted: stats['deleted'] += 1

                # We do not track deleted messages in our main array
                if not is_deleted:
                    # Attempt to parse the date string into a datetime object
                    dt = None
                    for fmt in ['%d/%m/%y %H:%M', '%d/%m/%Y %H:%M']:
                        try:
                            dt = datetime.strptime(date_str, fmt)
                            break
                        except ValueError:
                            pass

                    if dt:
                        current_msg = {'dt': dt, 'sender': sender, 'text': text, 'is_media': is_media}
                        if sender not in participants:
                            participants.append(sender)
                else:
                    current_msg = None
            except Exception:
                stats['system'] += 1
        else:
            # --- MULTI-LINE HANDLING ---
            # If a line doesn't start with a date, it belongs to the previous message
            if current_msg and not current_msg['is_media']:
                current_msg['text'] += " " + line

    # Catch the very last message in the file
    if current_msg:
        messages.append(current_msg)

    return messages, participants, stats

## Features 2, 3 & 5: The Analytics Engine
With the data cleaned and structured into a list of dictionaries, we extract key metrics in a single pass ($O(n)$ time complexity). We calculate message volume, vocabulary frequencies, and response behaviors.

In [24]:

# 3. METRICS & VOCABULARY ENGINE (UPDATED)

def get_longest_silence(active_days, start_date, total_days):
    """Calculates the maximum consecutive days a user sent 0 messages."""
    if not active_days: return total_days
    all_days = [start_date + timedelta(days=x) for x in range(total_days)]
    max_streak, current_streak = 0, 0

    for d in all_days:
        if d not in active_days:
            current_streak += 1
            max_streak = max(max_streak, current_streak)
        else:
            current_streak = 0
    return max_streak

def compute_metrics(messages, participants):
    total_msgs = len(messages)
    start_date = messages[0]['dt'].date()
    end_date = messages[-1]['dt'].date()
    total_days = (end_date - start_date).days + 1

    user_stats = {p: {
        'count': 0, 'words': 0, 'night_msgs': 0, 'drama_msgs': 0,
        'laughs': 0, 'questions': 0, 'caring': 0, 'links': 0,
        'active_days': set(), 'response_times': [], 'bursts': []
    } for p in participants}

    word_freq = {}
    day_counts = {}
    hour_counts = {h: 0 for h in range(24)}

    current_burst_sender = None
    current_burst_count = 0

    for i, m in enumerate(messages):
        sender, dt, text = m['sender'], m['dt'], m['text']
        stats = user_stats[sender]

        # 1. Basic Volume Tracking
        stats['count'] += 1
        stats['active_days'].add(dt.date())

        day_str = dt.strftime('%Y-%m-%d')
        day_counts[day_str] = day_counts.get(day_str, 0) + 1
        hour_counts[dt.hour] += 1

        # 2. Spammer Tracking
        if sender == current_burst_sender:
            current_burst_count += 1
        else:
            if current_burst_sender:
                user_stats[current_burst_sender]['bursts'].append(current_burst_count)
            current_burst_sender = sender
            current_burst_count = 1

        # 3. Response Time Tracking (gap between different senders)
        if i > 0 and messages[i-1]['sender'] != sender:
            diff = (dt - messages[i-1]['dt']).total_seconds() / 60.0
            stats['response_times'].append(diff)

        # 4. Text Processing
        if not m['is_media']:
            clean_text = text.translate(str.maketrans('', '', string.punctuation)).lower()
            words = clean_text.split()
            stats['words'] += len(words)

            # Archetype condition checks
            if dt.hour >= 23 or dt.hour < 5: stats['night_msgs'] += 1
            if (text.isupper() and len(text) > 3) or text.count('!') >= 2: stats['drama_msgs'] += 1
            if text.strip().endswith('?'): stats['questions'] += 1
            if 'http' in text or 'www' in text: stats['links'] += 1

            for w in words:
                if w in LAUGH_KEYWORDS: stats['laughs'] += 1
                if w in CARING_KEYWORDS: stats['caring'] += 1
                if w not in STOP_WORDS and len(w) > 2:
                    word_freq[w] = word_freq.get(w, 0) + 1

    if current_burst_sender:
        user_stats[current_burst_sender]['bursts'].append(current_burst_count)

    # 5. Finalize Averages and Streaks (This is what prevents the KeyError which i got the most!)
    for p, stats in user_stats.items():
        if stats['response_times']:
            stats['avg_response'] = sum(stats['response_times']) / len(stats['response_times'])
        else:
            stats['avg_response'] = 0.0

        stats['longest_silence'] = get_longest_silence(stats['active_days'], start_date, total_days)

    top_words = sorted(word_freq.items(), key=lambda x: x[1], reverse=True)[:10]
    busiest_day = sorted(day_counts.items(), key=lambda x: x[1], reverse=True)[0]
    busiest_hour = sorted(hour_counts.items(), key=lambda x: x[1], reverse=True)[0]

    return {
        'total_msgs': total_msgs, 'total_days': total_days,
        'start_date': start_date, 'end_date': end_date,
        'user_stats': user_stats, 'top_words': top_words,
        'busiest_day': busiest_day, 'busiest_hour': busiest_hour
    }

## Feature 4 & 7: The Heatmap & Archetypes
Here we utilize `NumPy` to generate an Activity Heatmap matrix (Rows = Users, Columns = Hours of the day). We then pass our calculated metrics through a rules engine to assign exclusive behavioural archetypes to each participant.

In [25]:

# 4. NUMPY HEATMAP & ARCHETYPE ASSIGNMENT

def build_heatmap(messages, participants):
    # Initialize a 2D NumPy array with zeros (Rows = Participants, Cols = 24 Hours)
    heatmap = np.zeros((len(participants), 24), dtype=int)

    for m in messages:
        # to Map the sender to their row index, and the hour to the column index
        p_idx = participants.index(m['sender'])
        hour = m['dt'].hour
        heatmap[p_idx, hour] += 1

    return heatmap

def get_longest_silence(active_days, start_date, total_days):
    """Calculates the maximum consecutive days a user sent zero messages."""
    if not active_days: return total_days
    all_days = [start_date + timedelta(days=x) for x in range(total_days)]
    max_streak, current_streak = 0, 0

    for d in all_days:
        if d not in active_days:
            current_streak += 1
            max_streak = max(max_streak, current_streak)
        else:
            current_streak = 0
    return max_streak

def assign_archetypes(user_stats, total_days, start_date):
    archetypes = {}

    for p, s in user_stats.items():
        tot = s['count']
        if tot == 0: continue

        # to Calculate averages for rule thresholds
        avg_burst = sum(s['bursts']) / len(s['bursts']) if s['bursts'] else 0
        avg_words = s['words'] / tot
        silent_days_streak = get_longest_silence(s['active_days'], start_date, total_days)

        # EXCLUSIVE ASSIGNMENT LOGIC
        # The if-elif chain ensures each participant receives exactly one archetype
        if silent_days_streak / total_days > 0.60:
            archetypes[p] = "👻 THE GHOST"
        elif avg_burst > 3:
            archetypes[p] = "🌪️ THE SPAMMER"
        elif s['night_msgs'] / tot > 0.60:
            archetypes[p] = "🦉 THE NIGHT OWL"
        elif s['drama_msgs'] / tot > 0.30:
            archetypes[p] = "🎭 THE DRAMA QUEEN"
        elif avg_words > 30:
            archetypes[p] = "📚 THE STORYTELLER"
        elif s['questions'] / tot > 0.25:
            archetypes[p] = "❓ THE QUESTION MASTER"
        elif s.get('links', 0) > 10:           # <--- CUSTOM ARCHETYPE the linker who sends the links to the chat
            archetypes[p] = "🔗 THE LINKER"
        elif s['caring'] > 10:
            archetypes[p] = "❤️ THE GROUP MOM"
        elif s['laughs'] / tot > 0.10:
            archetypes[p] = "😂 THE COMEDIAN"
        else:
            archetypes[p] = "🚶 THE NORMIE"

    return archetypes

## Feature 8: The Terminal UI Report
Formatting is graded. This function takes all calculated data structures and renders them using `f-strings` and ASCII characters to simulate a polished dashboard directly in the console output.

In [26]:

# 5. TERMINAL REPORT RENDERER

def print_report(data, participants, heatmap, archetypes, parse_stats):
    print("╔" + "═"*60 + "╗")
    print(f"║{'GROUPDNA REPORT':^60}║")
    print("╚" + "═"*60 + "╝")

    print(f"\n📊 OVERVIEW")
    print(f"   Period: {data['start_date']} to {data['end_date']} ({data['total_days']} days)")
    print(f"   Total Messages: {data['total_msgs']}")
    print(f"   Edge Cases Handled: {parse_stats['system']} System | {parse_stats['media']} Media | {parse_stats['deleted']} Deleted")

    print(f"\n🏆 TOP CONTRIBUTORS")
    ranked = sorted(data['user_stats'].items(), key=lambda x: x[1]['count'], reverse=True)
    for i, (p, s) in enumerate(ranked, 1):
        print(f"   {i}. {p:<15} {s['count']:>4} msgs")

    print(f"\n⏰ ACTIVITY PEAKS")
    print(f"   Busiest Day : {data['busiest_day'][0]} ({data['busiest_day'][1]} msgs)")
    print(f"   Busiest Hour: {data['busiest_hour'][0]:02d}:00 - {data['busiest_hour'][0]:02d}:59 ({data['busiest_hour'][1]} msgs)")

    print(f"\n🔥 TOP WORDS")
    if data['top_words']:
        max_c = data['top_words'][0][1]
        for word, count in data['top_words']:
            bar = "█" * int((count / max_c) * 15)
            print(f"   {word:<10} | {bar} {count}")

    # ---> NEW SECTION ADDED HERE <---
    print(f"\n⚡ HABITS & STREAKS")
    for p, s in ranked:
        print(f"   {p:<15} Avg Response: {s['avg_response']:>5.1f} mins | Longest Silence: {s['longest_silence']:>2} days")

    print(f"\n🎭 PERSONALITY ARCHETYPES")
    for p, arch in archetypes.items():
        print(f"   {p:<15} -> {arch}")

    print(f"\n📈 ACTIVITY HEATMAP (Rows=Users, Cols=Hours 0-23)")
    print("   " + " "*15 + "00 03 06 09 12 15 18 21")
    for i, p in enumerate(participants):
        row = f"   {p:<15} "
        max_v = np.max(heatmap[i]) if np.max(heatmap[i]) > 0 else 1

        for h in range(24):
            val = heatmap[i, h] / max_v
            if val == 0: char = ' '
            elif val < 0.25: char = '░'
            elif val < 0.50: char = '▒'
            elif val < 0.75: char = '▓'
            else: char = '█'

            row += char + (" " if (h+1)%3==0 else "")
        print(row)
    print("═"*62)

## Execution Pipeline
Uploaded the `hostel_bois.txt` dataset to the Colab files directory before running this final cell. This block sequentially calls our functions, chaining the outputs together.

In [27]:

# 6. EXECUTION PIPELINE

# the official project dataset
file_path = "hostel_bois.txt"

# Step 1: Parsed the raw text file
messages, participants, parse_stats = parse_chat(file_path)

# Step 2: Computed core metrics, word frequencies, response times, and streaks
data_metrics = compute_metrics(messages, participants)

# Step 3: Generated the NumPy activity matrix
heatmap_matrix = build_heatmap(messages, participants)

# Step 4: Run the exclusive archetype logic
archetypes = assign_archetypes(data_metrics['user_stats'], data_metrics['total_days'], data_metrics['start_date'])

# Step 5: Rendered the formatted output to the terminal
print_report(data_metrics, participants, heatmap_matrix, archetypes, parse_stats)

╔════════════════════════════════════════════════════════════╗
║                      GROUPDNA REPORT                       ║
╚════════════════════════════════════════════════════════════╝

📊 OVERVIEW
   Period: 2024-04-01 to 2024-05-30 (60 days)
   Total Messages: 3159
   Edge Cases Handled: 4 System | 32 Media | 15 Deleted

🏆 TOP CONTRIBUTORS
   1. Rahul            947 msgs
   2. Priya            716 msgs
   3. Neha             632 msgs
   4. Aman             488 msgs
   5. Karan            352 msgs
   6. Vikas             24 msgs

⏰ ACTIVITY PEAKS
   Busiest Day : 2024-05-04 (76 msgs)
   Busiest Hour: 18:00 - 18:59 (246 msgs)

🔥 TOP WORDS
   guys       | ███████████████ 318
   about      | ████████████ 274
   hai        | ████████████ 268
   today      | ████████████ 257
   his        | ██████████ 217
   which      | █████████ 202
   everyone   | ████████ 187
   telling    | ████████ 179
   bhai       | ███████ 160
   one        | ███████ 157

⚡ HABITS & STREAKS
   Rahul           A

# AI-assisted: I used an AI(gemini) to help structure the boilerplate parsing logic and to figure out the syntax for the NumPy heatmap rendering, but customized the rules and UI. also used it to manage the errors when got frustated